In [ ]:
import re
import sys
import numpy as np
from pathlib import Path as ph
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [ ]:
# Add parent directory to sys.path
sys.path.append(str(ph().resolve().parent))
from src.functions.runtime import from_file, towa_file, get_directory_files

In [ ]:
runtime_path_inp = input("Enter the runtime path ('same','<path>'): ").strip().lower()

In [ ]:
########################
# Runtime variables
########################

if runtime_path_inp == "same":
    runtime_path = "."
else:
    runtime_path = runtime_path_inp

configs_path = f"{runtime_path}/configs"
tokenizers_path = f"{runtime_path}/tokenizers"
inputs_path = f"{runtime_path}/inputs"
outputs_path = f"{runtime_path}/outputs"
models_path = f"{runtime_path}/models"
charts_path = f"{runtime_path}/charts"
statistics_path = f"{runtime_path}/statistics"

In [ ]:
def process_visualize_file_components(combined):
    statistic_path = f"{statistics_path}/statistic_experiments.csv"
    statistic = from_file(statistic_path, "csv")

    n_layers = [4,8,16]

    # Extract header and data
    header = statistic[0]
    header[0] = header[0].lstrip('\ufeff').lstrip('\ufeff') # Remove BOM if present
    data = statistic[1:]

    # Find columns index    
    w_col_a = header.index("c_device")
    w_col_b = header.index("n_layers")
    x_col_a = header.index("baby's brain")
    x_col_b = header.index("c_attention")
    x_col_c = header.index("c_network")
    x_col_d = header.index("total_time_execution")
    y_col = header.index("inference_quality_execution")

    if combined:
        fig, axes = plt.subplots(len(n_layers), 1, figsize=(16, 8 * len(n_layers)))
        if len(n_layers) == 1:
            axes = [axes]

    for idx, n_layer in enumerate(n_layers):
        x_axes = []
        y_axes = []
        colors = []
        x_part2_list = []
        for row in data:
            # Filter data
            if str(row[w_col_a]) == "gpu" and int(row[w_col_b]) == n_layer:
                x_part1 = str(row[x_col_a])
                x_part2 = f"{str(row[x_col_b])}\n{str(row[x_col_c])}\n{(int(row[x_col_d])/3600):.0f}h"
                x_axes.append(x_part1)
                y_axes.append(float(row[y_col].replace("/100", "")))
                x_part2_list.append(x_part2)
                if "finetuned" in x_part1:
                    colors.append("red")
                else:
                    colors.append("green")
        
        if combined:
            ax = axes[idx]
        else:
            fig, ax = plt.subplots(figsize=(16, 8))
        
        bars = ax.bar(x_axes, y_axes, color=colors, edgecolor='black')
        legend_handles = [
            Patch(color='green', label='baseline'),
            Patch(color='red', label='finetuned')
        ]
        ax.legend(handles=legend_handles)
        for bar, label in zip(bars, x_part2_list):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() / 2,
                label,
                ha='center',
                va='center',
                rotation=0,
                color='white',
                fontsize=8,
                fontweight='bold'
            )
        ax.set_title(f"Model Quality for n_layers={n_layer}")
        ax.set_ylabel("Quality")
        ax.set_xlabel("Model")
        ax.grid(True)
        ax.set_xticks(range(len(x_axes)))
        ax.set_xticklabels(x_axes, rotation=90, ha='center')

        if not combined:
            plt.tight_layout()
            plt.show()
            plt.close()

    if combined:
        for ax in axes:
            ax.set_xlabel("Model")
        plt.tight_layout()
        plt.savefig(f"{statistics_path}/statistic_experiments_bars.png", bbox_inches='tight')
        plt.show()
        plt.close()

In [ ]:
process_visualize_file_components(False)